The original issue was reported in https://its.cern.ch/jira/browse/ATLINFR-5806. 

```text
I've done a local build with gcc15 + onnxruntime 1.24.2 + cuda 13.1 - 
the build itself works fine for AthenaExternals + Athena (expect TrigInDetCUDA) 
but there are FPEs similarly as reported before in several CITests for derivations 
and a few new ones in reco

DerivationRun2Data_PHYS/DerivationRun2Data_PHYS.log:ERROR       22  FtagNN_AntiKt10UFOCSSKSoftDropBeta100Zcut10Jets_InDetTrackParticles_JetCalibTools_CalibArea-00-04-83_CalibrationFactors_bbJESJMS_calibFactors_R22_MC20MC23_CSSKUFO_bJR10v01_20250212_STANDARD_Jet
```

To reproduce the issue
```bash
$ ssh mypowerfulbuildmachine.cern.ch
$ setupATLAS
$ mkdir mybuild
$ lsetup git
$ git clone https://gitlab.cern.ch/atlas/athena.git
$ asetup none,gcc13,opt,cmakesetup --cmakeversion=4.0.1
$ ./athena/Projects/Athena/build_externals.sh -c -t Release -x -DATLAS_ONNXRUNTIME_SOURCE="URL;https://github.com/microsoft/onnxruntime/releases/download/v1.22.0/onnxruntime-linux-x64-1.22.0.tgz" > external.log 2>&1  &
$./athena/Projects/Athena/build.sh -cmi -t Release -x -DATLAS_ENABLE_CI_TESTS=TRUE > build.log 2>&1 &
$ cd build/build/Athena
$ source $HOME/mybuild/athena/Projects/Athena/build_env.sh 
$ ctest -j64
```



In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import onnx
import onnxruntime as ort
import numpy as np
from utils import inspect


print(ort.__version__)

1.22.0


In [ ]:
# model_filename = "../models/bbJESJMS_calibFactors_R22_MC20MC23_CSSKUFO_bJR10v01_20250212.onnx"
model_filename = "https://atlas-groupdata.web.cern.ch/atlas-groupdata/JetCalibTools/CalibArea-00-04-83/CalibrationFactors/bbJESJMS_calibFactors_R22_MC20MC23_CSSKUFO_bJR10v01_20250212.onnx"

In [14]:
inspect(model_filename)

Input names:  ['jet_features', 'track_features', 'flow_features'] with shapes:  [[1, 3], ['n_tracks', 24], ['n_flow', 4]]
Output names:  ['bJR10v01_mass', 'bJR10v01_pt'] with shapes:  [[], []]


In [33]:
def apply(model_path):
    n_track_features = 24
    n_flow_features = 4

    jet_features = np.array([[85507.8, -3.05748, 1.23]], dtype=np.float32)
    # track_features = np.random.rand(10, 24).astype(np.float32)
    # flow_features = np.random.rand(3, 4).astype(np.float32)

    track_features = np.array([]*n_track_features, dtype=np.float32)
    track_features = track_features.reshape(-1, n_track_features)

    flow_features = np.array([]*n_flow_features, dtype=np.float32)
    flow_features = flow_features.reshape(-1, n_flow_features)

    print("jet feature shape:", jet_features.shape)
    print("track feature shape:", track_features.shape)
    print("flow feature shape:", flow_features.shape)

    model = onnx.load(model_path)
    onnx.checker.check_model(model)
    # onnx.helper.printable_graph(model.graph)

    sess_opts = ort.SessionOptions()
    sess_opts.enable_mem_reuse = False
    ort_sess = ort.InferenceSession(model_path, sess_opts=sess_opts)
    input_names = [x.name for x in ort_sess.get_inputs()]
    output_names = [x.name for x in ort_sess.get_outputs()]

    sess_opts = ort.SessionOptions()
    results = ort_sess.run(output_names, {input_names[0]: jet_features, input_names[1]: track_features, input_names[2]: flow_features})
    return results

In [34]:
apply(model_filename)

jet feature shape: (1, 3)
track feature shape: (0, 24)
flow feature shape: (0, 4)


[array(0.6305852, dtype=float32), array(150916.56, dtype=float32)]